<a href="https://colab.research.google.com/github/asegura4488/EDCO_IntroCienciaDatos/blob/main/Semana3/Pandas_EstudioExploratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Para dividir los datos y construir el modelo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Métricas de evaluación
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    RocCurveDisplay,
    precision_recall_curve,
    average_precision_score
)

# Para que pandas muestre más columnas cuando inspeccionamos el DataFrame
pd.set_option('display.max_columns', 50)

In [3]:
# Carpeta de trabajo
os.chdir("/content/drive/MyDrive/EDCO_IntroCienciaDatos/Semana3")

# Leemos la base completa.
# Es mejor conservar todas las columnas al inicio y seleccionar después
# cuáles necesitaremos para cada análisis.
df = pd.read_csv("Datos/muestra.csv")

print("Dimensiones de la base:", df.shape)
df.head()

Dimensiones de la base: (300, 23)


,fecha reporte web,ID de caso,Fecha de notificación,Código DIVIPOLA departamento,Nombre departamento,Código DIVIPOLA municipio,Nombre municipio,Edad,Unidad de medida de edad,Sexo,Tipo de contagio,Ubicación del caso,Estado,Código ISO del país,Nombre del país,Recuperado,Fecha de inicio de síntomas,Fecha de muerte,Fecha de diagnóstico,Fecha de recuperación,Tipo de recuperación,Pertenencia étnica,Nombre del grupo étnico
0,2021-04-23 00:00:00,"2,731,846",2021-04-20 00:00:00,11,BOGOTA,"11,001",BOGOTA,26,1,F,Comunitaria,Casa,Leve,NaN,NaN,Recuperado,2021-04-19 00:00:00,NaN,2021-04-21 00:00:00,2021-05-04 00:00:00,PCR,6,NaN
1,2021-03-31 00:00:00,"2,398,518",2021-03-18 00:00:00,23,CORDOBA,"23,001",MONTERIA,18,1,M,Relacionado,Casa,Leve,NaN,NaN,Recuperado,2021-03-15 00:00:00,NaN,2021-03-29 00:00:00,2021-04-01 00:00:00,Tiempo,6,NaN
2,2021-10-18 00:00:00,"4,981,819",2021-10-03 00:00:00,11,BOGOTA,"11,001",BOGOTA,21,1,F,Relacionado,Casa,Leve,NaN,NaN,Recuperado,2021-09-29 00:00:00,NaN,2021-10-14 00:00:00,2021-10-20 00:00:00,Tiempo,6,NaN
3,2021-02-02 00:00:00,"2,108,069",2021-01-30 00:00:00,11,BOGOTA,"11,001",BOGOTA,39,1,F,Relacionado,Casa,Leve,NaN,NaN,Recuperado,2021-01-28 00:00:00,NaN,2021-01-31 00:00:00,2021-02-11 00:00:00,Tiempo,6,NaN
4,2022-01-11 00:00:00,"5,367,282",2021-12-28 00:00:00,11,BOGOTA,"11,001",BOGOTA,36,1,F,Relacionado,Casa,leve,NaN,NaN,Recuperado,2021-12-24 00:00:00,NaN,2022-01-08 00:00:00,2022-01-12 00:00:00,Tiempo,6,NaN


In [4]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

print("\nColumnas disponibles:")
for columna in df.columns:
    print("-", columna)

Número de filas: 300
Número de columnas: 23

Columnas disponibles:
- fecha reporte web
- ID de caso
- Fecha de notificación
- Código DIVIPOLA departamento
- Nombre departamento
- Código DIVIPOLA municipio
- Nombre municipio
- Edad
- Unidad de medida de edad
- Sexo
- Tipo de contagio
- Ubicación del caso
- Estado
- Código ISO del país
- Nombre del país
- Recuperado
- Fecha de inicio de síntomas
- Fecha de muerte
- Fecha de diagnóstico
- Fecha de recuperación
- Tipo de recuperación
- Pertenencia étnica
- Nombre del grupo étnico


In [5]:
# Limpieza -> valores faltantes, filas repetidas.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   fecha reporte web             300 non-null    object 
 1   ID de caso                    300 non-null    object 
 2   Fecha de notificación         300 non-null    object 
 3   Código DIVIPOLA departamento  300 non-null    object 
 4   Nombre departamento           300 non-null    object 
 5   Código DIVIPOLA municipio     300 non-null    object 
 6   Nombre municipio              300 non-null    object 
 7   Edad                          300 non-null    int64  
 8   Unidad de medida de edad      300 non-null    int64  
 9   Sexo                          300 non-null    object 
 10  Tipo de contagio              300 non-null    object 
 11  Ubicación del caso            298 non-null    object 
 12  Estado                        298 non-null    object 
 13  Códig

In [7]:
# Miremos las variables numericas
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Edad,300.0,40.816667,19.291369,1.0,27.0,38.5,53.0,92.0
Unidad de medida de edad,300.0,1.003333,0.057735,1.0,1.0,1.0,1.0,2.0
Código ISO del país,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Nombre del país,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pertenencia étnica,300.0,5.916667,0.586688,1.0,6.0,6.0,6.0,6.0


In [8]:
# Miremos las variables categoricas
df.describe(include='object').T

,count,unique,top,freq
fecha reporte web,300,206,2022-07-21 00:00:00,5
ID de caso,300,300,"1,723,385",1
Fecha de notificación,300,220,2021-06-15 00:00:00,6
Código DIVIPOLA departamento,300,29,11,103
Nombre departamento,300,29,BOGOTA,103
Código DIVIPOLA municipio,300,83,"11,001",103
Nombre municipio,300,83,BOGOTA,103
Sexo,300,2,F,162
Tipo de contagio,300,2,Comunitaria,214
Ubicación del caso,298,2,Casa,289


In [9]:
# Cantidad de valores faltantes por variable
faltantes = df.isna().sum().sort_values(ascending=False)
faltantes_df = pd.DataFrame({
    'faltantes': faltantes,
    'porcentaje': (faltantes / df.shape[0]) * 100
})
faltantes_df

,faltantes,porcentaje
Código ISO del país,300,100.000000
Nombre del país,300,100.000000
Nombre del grupo étnico,296,98.666667
Fecha de muerte,289,96.333333
Fecha de inicio de síntomas,22,7.333333
Fecha de recuperación,11,3.666667
Tipo de recuperación,11,3.666667
Estado,2,0.666667
Recuperado,2,0.666667
Ubicación del caso,2,0.666667


In [10]:
# Revisamos los registros duplicados
df.duplicated().sum()

np.int64(0)

In [12]:
# Hacemos una copia -> queremos mantener la informacion original
datos = df.copy()

# Pequeña ingenieria de caracteristicas

columnas_texto = [
    'Sexo',
    'Tipo de contagio',
    'Estado',
    'Recuperado',
    'Ubicación del caso'
]

In [15]:
# Quitar los espacios en blanco
for col in columnas_texto:
  if col in datos.columns:
    datos[col] = datos[col].astype('string').str.strip()

In [16]:
# Limpieza
datos['Estado'] = datos['Estado'].str.capitalize()
# evitar tener Leve vs leve
datos['Estado'].value_counts(dropna=False)

,count
Estado,
Leve,289
Fallecido,9
<NA>,2


In [ ]:
# Analisis exploratorio de datos (EDA)
